# Training Bronze ingest/routing

Writes a training Bronze table with the same columns as `bronze_vehicle_positions_valid`.

Unlike the production route notebook, this branch does not split bad rows into quarantine. It keeps parse flags in the same Bronze-valid-shaped table so AIOps can build quality features from the unfiltered stream.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

TRAINING_BRONZE_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.training_bronze_hsl_vehicle_position"
TRAINING_BRONZE_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/bronze/hsl_vehicle_position"
CHECKPOINT_BRONZE_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/training_data/checkpoints/bronze_hsl_vehicle_position"

EVENTHUB_NAMESPACE = "hant-event-hub"
EVENTHUB_NAME = "vehicle-position-events"
BOOTSTRAP_SERVERS = f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093"

SECRET_SCOPE = "eventhub-scope"
SAS_KEY_NAME_SECRET = "sas-key-name"
SAS_KEY_SECRET = "sas-key"

SAS_KEY_NAME = dbutils.secrets.get(scope=SECRET_SCOPE, key=SAS_KEY_NAME_SECRET)
SAS_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key=SAS_KEY_SECRET)

KAFKA_SASL_JAAS = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="Endpoint=sb://{EVENTHUB_NAMESPACE}.servicebus.windows.net/;'
    f'SharedAccessKeyName={SAS_KEY_NAME};'
    f'SharedAccessKey={SAS_KEY};'
    f'EntityPath={EVENTHUB_NAME}";'
)

STARTING_OFFSETS = "latest"
FAIL_ON_DATA_LOSS = "false"
MAX_OFFSETS_PER_TRIGGER = 5000
TRIGGER_INTERVAL = "10 seconds"

RESET_TABLE = False
RESET_CHECKPOINT = False

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

for q in spark.streams.active:
    if q.name == "training_bronze_hsl_vehicle_position":
        q.stop()

if RESET_TABLE:
    spark.sql(f"DROP TABLE IF EXISTS {TRAINING_BRONZE_TABLE}")
    dbutils.fs.rm(TRAINING_BRONZE_PATH, True)

if RESET_CHECKPOINT:
    dbutils.fs.rm(CHECKPOINT_BRONZE_PATH, True)


In [ ]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TRAINING_BRONZE_TABLE} (
    topic STRING,
    partition INT,
    offset BIGINT,
    eventhub_enqueued_ts TIMESTAMP,
    message_key STRING,
    raw_json STRING,
    bronze_ingest_ts TIMESTAMP,
    ingest_date DATE,
    parse_ok BOOLEAN,
    parse_error STRING,
    source STRING,
    producer_ingest_ts_utc STRING,
    mqtt_topic STRING,
    mqtt_qos INT,
    mqtt_retain BOOLEAN,
    event_type STRING,
    transport_mode STRING
)
USING DELTA
PARTITIONED BY (ingest_date)
LOCATION "{TRAINING_BRONZE_PATH}"
''')


In [ ]:
envelope_schema = T.StructType([
    T.StructField("ingest_ts_utc", T.StringType(), True),
    T.StructField("source", T.StringType(), True),
    T.StructField("mqtt", T.StructType([
        T.StructField("topic", T.StringType(), True),
        T.StructField("qos", T.IntegerType(), True),
        T.StructField("retain", T.BooleanType(), True),
    ]), True),
    T.StructField("topic_parsed", T.StructType([
        T.StructField("event_type", T.StringType(), True),
        T.StructField("transport_mode", T.StringType(), True),
    ]), True),
])

raw_kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", EVENTHUB_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", KAFKA_SASL_JAAS)
    .option("startingOffsets", STARTING_OFFSETS)
    .option("failOnDataLoss", FAIL_ON_DATA_LOSS)
    .option("maxOffsetsPerTrigger", MAX_OFFSETS_PER_TRIGGER)
    .load()
)

training_bronze_stream = (
    raw_kafka_df
    .select(
        F.col("topic").cast("string").alias("topic"),
        F.col("partition").cast("int").alias("partition"),
        F.col("offset").cast("bigint").alias("offset"),
        F.col("timestamp").alias("eventhub_enqueued_ts"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_json"),
        F.current_timestamp().alias("bronze_ingest_ts"),
    )
    .withColumn("ingest_date", F.to_date("bronze_ingest_ts"))
    .withColumn("parsed_envelope", F.from_json(F.col("raw_json"), envelope_schema))
    .withColumn(
        "parse_ok",
        F.col("raw_json").isNotNull()
        & F.col("parsed_envelope.source").isNotNull()
        & F.col("parsed_envelope.mqtt.topic").isNotNull()
    )
    .withColumn(
        "parse_error",
        F.when(F.col("raw_json").isNull(), F.lit("raw_json_is_null"))
        .when(F.col("parsed_envelope.source").isNull(), F.lit("invalid_envelope_json"))
        .when(F.col("parsed_envelope.mqtt.topic").isNull(), F.lit("missing_mqtt_topic"))
        .otherwise(F.lit(None).cast("string"))
    )
    .withColumn("source", F.col("parsed_envelope.source"))
    .withColumn("producer_ingest_ts_utc", F.col("parsed_envelope.ingest_ts_utc"))
    .withColumn("mqtt_topic", F.col("parsed_envelope.mqtt.topic"))
    .withColumn("mqtt_qos", F.col("parsed_envelope.mqtt.qos"))
    .withColumn("mqtt_retain", F.col("parsed_envelope.mqtt.retain"))
    .withColumn("event_type", F.col("parsed_envelope.topic_parsed.event_type"))
    .withColumn("transport_mode", F.col("parsed_envelope.topic_parsed.transport_mode"))
    .select(
        "topic", "partition", "offset", "eventhub_enqueued_ts", "message_key", "raw_json",
        "bronze_ingest_ts", "ingest_date", "parse_ok", "parse_error", "source",
        "producer_ingest_ts_utc", "mqtt_topic", "mqtt_qos", "mqtt_retain",
        "event_type", "transport_mode",
    )
)

training_bronze_query = (
    training_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_BRONZE_PATH)
    .queryName("training_bronze_hsl_vehicle_position")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(TRAINING_BRONZE_TABLE)
)

print("Training Bronze stream started.")
print("  Query name:", training_bronze_query.name)
print("  Query ID  :", training_bronze_query.id)
